# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tanvir-Sheikh-R/From-flyrank-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import numpy as np
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"{len(df):,} rows | {df['client_id'].nunique()} clients | base rate: {df['is_declining_label'].mean():.3f}")

In [ ]:
NUMERIC_FEATURES = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
CATEGORICAL_FEATURES = ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

def build_features(frame):
    work = frame.copy()
    for col in NUMERIC_FEATURES:
        work[col] = pd.to_numeric(work[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    for col in CATEGORICAL_FEATURES:
        work[col] = work[col].fillna("unknown").astype(str)
    X_numeric = work[NUMERIC_FEATURES]
    X_categorical = pd.get_dummies(work[CATEGORICAL_FEATURES], dummy_na=False, dtype=float)
    X = pd.concat([X_numeric.reset_index(drop=True), X_categorical.reset_index(drop=True)], axis=1)
    return X, work

X, work = build_features(df)
y = df["is_declining_label"]
print("Feature matrix:", X.shape)

In [ ]:
def percentile_rank(s):
    return s.rank(method="average", pct=True).fillna(0)

visibility = percentile_rank(np.log1p(work["impressions_90d"]))
freshness_risk = percentile_rank(work["days_since_last_update"])
position_opportunity = (1 - percentile_rank(work["avg_position"].clip(1, 50))) * visibility * (work["avg_position"] > 0)
depth_gap = (1 - percentile_rank(work["word_count"])) * visibility
baseline_score = (0.40*visibility + 0.30*freshness_risk + 0.25*position_opportunity + 0.05*depth_gap).clip(0, 1)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print("Baseline score built (0.40 visibility + 0.30 freshness_risk + 0.25 position_opportunity + 0.05 depth_gap)")

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=work["client_id"]))

model = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                                class_weight="balanced_subsample", random_state=42, n_jobs=-1)
model.fit(X.iloc[train_idx], y.iloc[train_idx])
work["model_probability"] = model.predict_proba(X)[:, 1]
print("Model trained on client-holdout train split; probabilities scored for all rows.")

## 1. Ranked actions + reason codes

**The queue, in plain words:** blend the model's probability with the transparent baseline
score (70% model, 30% baseline — the model gets more weight because it's validated against the
baseline, but the baseline keeps the score from being a total black box). Every row gets a
reason code so a reviewer can see *why*, and a suggested action.

In [ ]:
work["baseline_score_norm"] = baseline_score
work["final_score"] = 100 * (0.70*work["model_probability"] + 0.30*work["baseline_score_norm"])

def reason_codes(row):
    reasons = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if row["trend_direction"].lower() == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if row["sessions_90d"] >= 30 and (
        (row["engagement_rate"] > 0 and row["engagement_rate"] < 30)
        or (row["scroll_rate"] > 0 and row["scroll_rate"] < 30)
    ):
        reasons.append("low_engagement_visible_page")
    if row["model_probability"] >= 0.65:
        reasons.append("model_decline_risk")
    return "|".join(reasons) or "general_refresh_review"

def suggested_action(reason_str):
    reasons = set(reason_str.split("|"))
    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"
    if "low_engagement_visible_page" in reasons:
        return "refresh_and_review_engagement"
    if {"model_decline_risk", "declining_with_demand", "stale_visible_page"} & reasons:
        return "refresh"
    return "monitor"

work["reason_codes"] = work.apply(reason_codes, axis=1)
work["suggested_action"] = work["reason_codes"].apply(suggested_action)

queue = work.sort_values("final_score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

show_cols = ["rank", "content_id", "final_score", "model_probability", "suggested_action", "reason_codes"]
print(queue[show_cols].head(15).to_string(index=False))
print(f"\nAction mix:\n{queue['suggested_action'].value_counts().to_string()}")

## 2. Intended use and limits

**Who uses this:** a content strategist or SEO editor with limited review time per week,
scanning the top of this queue to decide what to look at first.

**What it's for:** prioritizing *review*, not automating *action*. The output is "look at this
page next," never "change this page automatically."

**Where it stops being valid:**
- Any client with fewer than ~50 pages in the dataset — too little history for the model's
  signals (impressions, position trends) to mean much; fall back to the plain baseline rule.
- Pages younger than 90 days — excluded upstream already (`content_age_days >= 90` filter),
  since "declining" needs enough history to even be measurable.
- Any time period other than the one this was trained on — search behavior shifts, and this
  queue is only as fresh as its training data. See Section 4 for exactly when to distrust it.
- It never claims a refresh will improve performance — that requires an experiment, not this
  ranking.

In [ ]:
client_page_counts = work.groupby("client_id").size().sort_values()
thin_clients = client_page_counts[client_page_counts < 50]
print(f"Clients with fewer than 50 pages in this dataset: {len(thin_clients)} of {work['client_id'].nunique()}")
print("Queue rows belonging to thin clients should be treated as baseline-only, not model-trusted:")
thin_rows = queue[queue["client_id"].isin(thin_clients.index)]
print(f"  {len(thin_rows)} of {len(queue)} queue rows fall under this caveat")

## 3. Human review + the no-go list

**What a person must check before acting on any recommended page:**
- Is the traffic drop actually a content problem, or consolidation (a sibling page absorbed
  the demand), seasonality, or a SERP feature change? The model can't tell these apart.
- Does the page still match current search intent, or has the topic itself moved on?
- Is this a legally/compliance-sensitive page where any edit needs separate sign-off?

**The no-go list — never automate these directly from this queue:**
- Auto-publishing content changes based on `final_score` alone.
- Auto-pruning/deleting pages flagged `monitor` or low-confidence — deletion needs a human
  decision, always.
- Treating `model_decline_risk` as proof of a real problem for a single page in isolation —
  it's a ranking signal across many pages, not a certified diagnosis for one.

## 4. Monitoring / retrain triggers

Signs the recommendations have gone stale and the model needs a fresh training run:

- **Score drift:** the distribution of `final_score` shifts noticeably (e.g. median score
  moves by more than ~10 points) between runs on new data — the underlying pages or search
  landscape changed.
- **Action mix drift:** the proportion of `refresh` vs `monitor` actions swings sharply run to
  run, with no known cause (e.g. an algorithm update, a big content push).
- **Feature importance reshuffling:** if a re-trained model's top features change order
  substantially, the relationships in the data have shifted and old conclusions may not hold.
- **Time-based default:** retrain at least every 90 days regardless, since the label window
  itself is 90 days — recommendations older than that are working from a different world.

In [ ]:
current_summary = {
    "median_final_score": float(queue["final_score"].median()),
    "action_mix": queue["suggested_action"].value_counts(normalize=True).round(3).to_dict(),
    "top5_features_snapshot": (
        pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False).head(5).round(4).to_dict()
    ),
}
import json
print("Save this snapshot alongside the queue — compare it next time you retrain:")
print(json.dumps(current_summary, indent=2))

## 5. Exports for the paper

Write the final queue and the monitoring snapshot to `work/outputs/` — the capstone paper's
Results and Ranked Recommendations sections build directly on these files.

In [ ]:
os.makedirs("work/outputs", exist_ok=True)

out_cols = ["rank", "content_id", "client_id", "final_score", "model_probability",
            "baseline_score_norm", "reason_codes", "suggested_action",
            "is_declining_label", "impressions_90d", "avg_position", "ctr", "trend_direction"]
queue[out_cols].to_csv("work/outputs/action_playbook_queue.csv", index=False)
print("Wrote work/outputs/action_playbook_queue.csv")

with open("work/outputs/monitoring_snapshot.json", "w") as f:
    json.dump(current_summary, f, indent=2)
print("Wrote work/outputs/monitoring_snapshot.json")

print(f"\nFinal queue: {len(queue):,} rows | top score: {queue['final_score'].max():.1f} | "
      f"action mix: {dict(queue['suggested_action'].value_counts())}")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this yourself
      to confirm; I could not execute it in this sandbox (no dataset file here)**
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.